# TDA Mapper — Clustering Topológico de Anomalías FINANOM

Usa el algoritmo Mapper para encontrar la "forma" del espacio de anomalías:
qué subtipos existen, cómo se conectan y qué features los distinguen.

**Flujo:**
1. Jalamos solo las anomalías detectadas desde Azure SQL
2. Construimos features numéricas (score, monto, hora, severidad)
3. Reducimos a 2D con PCA (o UMAP si está instalado) como lente
4. Corremos Mapper con `kmapper`
5. Generamos el grafo interactivo en `output/mapper_graph.html`
6. Analizamos el perfil de cada nodo

In [6]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import kmapper as km
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model_final import db

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Cargar datos locales

Usamos los archivos parquet/joblib del modelo local — sin conexión a Azure SQL.

In [7]:
ROOT = Path().resolve().parent  # c:\Users\anton\FinAnom

df_full = pd.read_parquet(ROOT / "model_final" / "output" / "reporte_revision.parquet")
print(f"Shape: {df_full.shape}")
print(f"Columnas: {list(df_full.columns)}")

# Renombrar si hace falta
if "monto" in df_full.columns and "t_monto" not in df_full.columns:
    df_full = df_full.rename(columns={"monto": "t_monto"})
if "severidad" in df_full.columns and "severity" not in df_full.columns:
    df_full = df_full.rename(columns={"severidad": "severity"})

# Filtrar solo anomalías
anomaly_col = "is_anomaly" if "is_anomaly" in df_full.columns else "is_anomaly_if"
df = df_full[df_full[anomaly_col].astype(bool)].copy().reset_index(drop=True)
print(f"\nAnomалías cargadas: {len(df):,}")
df.head(3)

Shape: (1145526, 20)
Columnas: ['trace_row_id', 'trace_t_folio', 'trace_t_folio_ext', 'trace_t_referencia', 'trace_t_transaccion', 'trace_t_cve_res', 'trace_t_cuarto', 'trace_t_codigo', 'trace_t_timestamp', 'anomaly_score', 'anomaly_pct', 'score_samples', 'is_anomaly_if', 'rule_score', 'severidad', 'is_anomaly', 'tipo_inconsistencia', 'motivos', 'evidencia_shap', 'requiere_aprobacion']

Anomалías cargadas: 47,087


,trace_row_id,trace_t_folio,trace_t_folio_ext,trace_t_referencia,trace_t_transaccion,trace_t_cve_res,trace_t_cuarto,trace_t_codigo,trace_t_timestamp,anomaly_score,anomaly_pct,score_samples,is_anomaly_if,rule_score,severity,is_anomaly,tipo_inconsistencia,motivos,evidencia_shap,requiere_aprobacion
0,80,7574,0,3201M1,293,I 41880 1,3201,PROPTI,2021-06-28 05:31:00,0.469083,0.660257,-0.469083,False,60,ALTO,True,DUPLICADO | CONTEXTO_RESERVACION,"Alerta tx 293 (folio 7574, cód PROPTI, monto 1...",,True
1,289,7572,0,6109H2,502,I 41390 1,6109,RENHAB,2021-06-28 05:31:00,0.524121,0.914509,-0.524121,False,65,ALTO,True,DUPLICADO | MONTO_ATIPICO,"Alerta tx 502 (folio 7572, cód RENHAB, monto 6...",,True
2,330,7430,1,8106A1,773,I 7538 2,8106,PROPTI,2021-06-26 02:51:00,0.538359,0.940799,-0.538359,False,90,CRITICO,True,DUPLICADO | CANCELACION_SOSPECHOSA,"Alerta tx 773 (folio 7430, cód PROPTI, monto 2...",,True


In [8]:
ts = pd.to_datetime(df["trace_t_timestamp"].astype(str), errors="coerce")
df["hora"]       = ts.dt.hour
df["dia_semana"] = ts.dt.dayofweek
df["mes"]        = ts.dt.month

sev_map = {"ALTO": 2, "CRITICO": 3, "MEDIO": 1, "BAJO": 0}
df["severidad_num"] = df["severity"].map(sev_map).fillna(0)
df["cat_principal"] = df["tipo_inconsistencia"].fillna("Sin categoría").str.split(" | ").str[0]

FEATURES = ["anomaly_score", "rule_score", "hora", "dia_semana", "mes", "severidad_num"]
X_raw = df[FEATURES].fillna(0).values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Shape: {X.shape}")
print(f"\nCategorías principales:")
print(df["cat_principal"].value_counts().to_string())

Shape: (47087, 6)

Categorías principales:
cat_principal
DUPLICADO                 34358
MONTO_ATIPICO              3582
CANCELACION_SOSPECHOSA     3398
CONTEXTO_RESERVACION       2285
FUERA_DE_ESTANCIA          2045
ATIPICO_IF                 1404
SIGNO_CONTABLE                8
METODO_PAGO                   7


## 3. Lente: reducción dimensional

Usamos UMAP si está disponible (mejor topología), si no PCA.

In [9]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    lens = reducer.fit_transform(X)
    lens_name = "UMAP"
except ImportError:
    pca = PCA(n_components=2, random_state=42)
    lens = pca.fit_transform(X)
    lens_name = f"PCA (varianza explicada: {pca.explained_variance_ratio_.sum():.1%})"

print(f"Lente: {lens_name}")
print(f"Shape lente: {lens.shape}")

Lente: UMAP
Shape lente: (47087, 2)


## 4. Mapper

- `n_cubes`: resolución del cover (más alto = más nodos, más detalle)
- `perc_overlap`: overlap entre cubos (más alto = más conexiones)
- Clusterer: DBSCAN dentro de cada cubo

In [10]:
mapper = km.KeplerMapper(verbose=1)

graph = mapper.map(
    lens,
    X,
    clusterer=DBSCAN(eps=0.6, min_samples=3),
    cover=km.Cover(n_cubes=12, perc_overlap=0.5),
)

n_nodes  = len(graph["nodes"])
n_edges  = sum(len(v) for v in graph["links"].values())
print(f"\nNodos: {n_nodes}  |  Aristas: {n_edges}")

KeplerMapper(verbose=1)
Mapping on data shaped (47087, 6) using lens shaped (47087, 2)

Creating 144 hypercubes.

Created 1295 edges and 661 nodes in 0:00:17.949486.

Nodos: 661  |  Aristas: 1295


## 5. Visualización interactiva

In [11]:
# Coloreamos por anomaly_score promedio del nodo
html_path = str(OUTPUT_DIR / "mapper_graph.html")

mapper.visualize(
    graph,
    path_html=html_path,
    title="FINANOM — Mapa Topológico de Anomalías",
    color_values=df["anomaly_score"].values,
    color_function_name="Anomaly Score",
    node_color_function=["mean", "std", "max"],
)

print(f"Grafo guardado en: {html_path}")
print("Ábrelo en el browser para explorar el grafo interactivo.")

Wrote visualization to: output\mapper_graph.html
Grafo guardado en: output\mapper_graph.html
Ábrelo en el browser para explorar el grafo interactivo.


## 6. Perfil de nodos — ¿qué hay en cada cluster?

In [12]:
profiles = []
for node_id, member_idx in graph["nodes"].items():
    members = df.iloc[list(member_idx)]
    profiles.append({
        "nodo":           node_id,
        "n":              len(members),
        "score_mean":     round(members["anomaly_score"].mean(), 3),
        "rule_score_mean":round(members["rule_score"].mean(), 1),
        "hora_mean":      round(members["hora"].mean(), 1),
        "severidad_mean": round(members["severidad_num"].mean(), 2),
        "cat_top":        members["cat_principal"].mode()[0] if len(members) else "",
        "cats_unicas":    members["cat_principal"].nunique(),
    })

profiles_df = (
    pd.DataFrame(profiles)
    .sort_values("score_mean", ascending=False)
    .reset_index(drop=True)
)
print("Top 20 nodos por anomaly_score promedio:")
profiles_df.head(20)

Top 20 nodos por anomaly_score promedio:


,nodo,n,score_mean,rule_score_mean,hora_mean,severidad_mean,cat_top,cats_unicas
0,cube40_cluster13,4,0.656,95.0,11.5,3.0,MONTO_ATIPICO,1
1,cube28_cluster14,4,0.656,95.0,11.5,3.0,MONTO_ATIPICO,1
2,cube27_cluster4,4,0.656,95.0,11.5,3.0,MONTO_ATIPICO,1
3,cube39_cluster6,4,0.656,95.0,11.5,3.0,MONTO_ATIPICO,1
4,cube28_cluster35,4,0.650,145.0,20.0,3.0,DUPLICADO,1
5,cube40_cluster12,8,0.650,110.0,15.4,3.0,DUPLICADO,1
6,cube28_cluster13,8,0.650,110.0,15.4,3.0,DUPLICADO,1
7,cube41_cluster12,8,0.650,110.0,15.4,3.0,DUPLICADO,1
8,cube17_cluster16,4,0.650,145.0,20.0,3.0,DUPLICADO,1
9,cube16_cluster16,4,0.650,145.0,20.0,3.0,DUPLICADO,1


## 7. ¿Qué categorías dominan el grafo?

In [13]:
import matplotlib.pyplot as plt

# Distribución de categoría dominante por nodo
cat_dist = profiles_df["cat_top"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nodos por categoría dominante
cat_dist.plot(kind="bar", ax=axes[0], color="#dc2626")
axes[0].set_title("Nodos por categoría dominante")
axes[0].set_ylabel("Nodos")
axes[0].tick_params(axis="x", rotation=30)

# Score promedio por categoría
score_by_cat = profiles_df.groupby("cat_top")["score_mean"].mean().sort_values(ascending=False)
score_by_cat.plot(kind="bar", ax=axes[1], color="#1d4ed8")
axes[1].set_title("Anomaly score promedio por categoría")
axes[1].set_ylabel("Score")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "categorias.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en output/categorias.png")

Guardado en output/categorias.png


## 8. Explorar un nodo específico

In [14]:
nodo_idx = 0
nodo_id  = profiles_df.iloc[nodo_idx]["nodo"]

miembros = df.iloc[list(graph["nodes"][nodo_id])]
print(f"Nodo: {nodo_id}  ({len(miembros)} transacciones)")
print(f"Score promedio:      {miembros['anomaly_score'].mean():.3f}")
print(f"Rule score promedio: {miembros['rule_score'].mean():.1f}")
print(f"\nCategorías:")
print(miembros["cat_principal"].value_counts().to_string())
print(f"\nSeveridad:")
print(miembros["severity"].value_counts().to_string())
miembros[["trace_t_folio", "trace_t_cuarto", "trace_t_codigo", "trace_t_timestamp", "anomaly_score", "cat_principal"]].head(10)

Nodo: cube40_cluster13  (4 transacciones)
Score promedio:      0.656
Rule score promedio: 95.0

Categorías:
cat_principal
MONTO_ATIPICO    4

Severidad:
severity
CRITICO    4


,trace_t_folio,trace_t_cuarto,trace_t_codigo,trace_t_timestamp,anomaly_score,cat_principal
10629,58881,9205,XFAC,2023-09-02 11:12:00,0.664148,MONTO_ATIPICO
29801,58881,9205,XFAC,2023-09-02 11:12:00,0.673466,MONTO_ATIPICO
31265,74524,3216,XFAC,2023-10-28 12:20:00,0.648273,MONTO_ATIPICO
31692,74524,3216,XFAC,2023-10-28 12:20:00,0.638004,MONTO_ATIPICO
